In [1]:
pwd

'd:\\GIT_PROJECT\\ViPhoLM\\notebooks\\machine_translation'

In [2]:
cd "D:\GIT_PROJECT\ViPhoLM\" 

D:\GIT_PROJECT\ViPhoLM


In [11]:
import json
import os

DIR = r"D:\GIT_PROJECT\ViPhoLM\datasets\PhoMT"
json_paths = "train.json"
full_path = os.path.join(DIR, json_paths)
full_path

'D:\\GIT_PROJECT\\ViPhoLM\\datasets\\PhoMT\\train.json'

In [13]:
import json
from collections import Counter
import numpy as np

def build_simple_vocab(json_data, min_freq=2):
    counter = Counter()
    for item in json_data:
        # Tách chữ bằng dấu cách
        counter.update(item["english"].lower().split())
        counter.update(item["vietnamese"].lower().split())
    
    # Chỉ lấy những từ xuất hiện trên 'min_freq' lần để tránh rác
    words = [word for word, count in counter.items() if count >= min_freq]
    
    # Các token đặc biệt
    special_tokens = ["<pad>", "<bos>", "<eos>", "<unk>"]
    vocab = {word: i for i, word in enumerate(special_tokens + words)}
    
    return vocab

# Cách dùng:
with open(full_path, 'r', encoding='utf-8') as f:
    data = json.load(f)
word_2_id = build_simple_vocab(data)

In [14]:
from tqdm import tqdm

def convert_to_npy(json_data, word_2_id, save_prefix, max_len=128):
    num_samples = len(json_data)
    
    # Khởi tạo mảng trắng với ID của <pad> (thường là 0)
    pad_idx = word_2_id.get("<pad>", 0)
    bos_idx = word_2_id.get("<bos>", 1)
    eos_idx = word_2_id.get("<eos>", 2)
    unk_idx = word_2_id.get("<unk>", 3)
    
    en_npy = np.full((num_samples, max_len), pad_idx, dtype=np.int32)
    vi_npy = np.full((num_samples, max_len), pad_idx, dtype=np.int32)

    for idx, item in enumerate(tqdm(json_data, desc="Đang map Chữ -> Số")):
        # Xử lý tiếng Anh
        en_words = item["english"].lower().split()[:max_len-2]
        en_ids = [bos_idx] + [word_2_id.get(w, unk_idx) for w in en_words] + [eos_idx]
        en_npy[idx, :len(en_ids)] = en_ids

        # Xử lý tiếng Việt
        vi_words = item["vietnamese"].lower().split()[:max_len-2]
        vi_ids = [bos_idx] + [word_2_id.get(w, unk_idx) for w in vi_words] + [eos_idx]
        vi_npy[idx, :len(vi_ids)] = vi_ids

    # Lưu file
    np.save(f"{save_prefix}_en.npy", en_npy)
    np.save(f"{save_prefix}_vi.npy", vi_npy)
    
    # QUAN TRỌNG: Lưu lại bộ từ điển để sau này còn dịch ngược lại (số -> chữ)
    with open(f"{save_prefix}_vocab.json", 'w', encoding='utf-8') as f:
        json.dump(word_2_id, f, ensure_ascii=False)

    print(f"\nĐã xong! Tổng từ vựng: {len(word_2_id)}")

In [15]:
convert_to_npy(data, word_2_id, "pho_mt_data", max_len=128)

Đang map Chữ -> Số: 100%|██████████| 20000/20000 [00:00<00:00, 111014.46it/s]


Đã xong! Tổng từ vựng: 14416


In [5]:
from configs.utils import get_config
config = get_config("configs/Transformer_MT_phoneme_Wikilingual.yaml")
from vocabs.utils import preprocess_sentence

In [ ]:
import numpy as np
import json 
import os
from tqdm import tqdm

def preprocess_to_npy(json_data, save_prefix, vocab, max_len=512):
    # json_data: nên là list đã load từ json.load()
    num_samples = len(json_data)
    print(f"Tổng số mẫu: {num_samples}")

    en_path = f"{save_prefix}_en.npy"
    vi_path = f"{save_prefix}_vi.npy"

    # --- BƯỚC FIX LỖI WINDOWS: Tạo file rỗng có kích thước chuẩn ---
    for path, shape in [(en_path, (num_samples, max_len)), 
                        (vi_path, (num_samples, max_len, 3))]:
        # Tính toán dung lượng file cần thiết (số phần tử * 4 bytes của int32)
        filesize = np.prod(shape) * np.dtype('int32').itemsize
        
        print(f"Đang cấp phát không gian cho {path} ({filesize / 1e9:.2f} GB)...")
        with open(path, 'wb') as f:
            f.seek(filesize - 1)
            f.write(b'\0')

    # --- BƯỚC 2: Ánh xạ bộ nhớ với mode 'r+' (chỉnh sửa trên file đã có) ---
    en_memmap = np.memmap(en_path, dtype='int32', mode='r+', shape=(num_samples, max_len))
    vi_memmap = np.memmap(vi_path, dtype='int32', mode='r+', shape=(num_samples, max_len, 3))

    # Đặt giá trị mặc định ban đầu là PAD
    en_memmap[:] = vocab.pad_idx
    vi_memmap[:] = vocab.pad_idx

    # --- BƯỚC 3: Tokenize và đổ dữ liệu vào ---
    for idx, item in enumerate(tqdm(json_data, desc="Đang chuyển đổi")):
        # English
        en_ids = vocab.encode_sentence_english(item["english"])[:max_len]
        en_memmap[idx, :len(en_ids)] = en_ids

        # Vietnamese
        vi_ids = vocab.encode_caption_vietnamese(item["vietnamese"])[:max_len]
        for s_idx, phoneme_group in enumerate(vi_ids):
            vi_memmap[idx, s_idx, :] = phoneme_group

    # Lưu và đóng
    en_memmap.flush()
    vi_memmap.flush()
    del en_memmap, vi_memmap # Giải phóng object để đóng file hoàn toàn
    print(f"\nThành công! File đã sẵn sàng tại: {en_path}")

: 

: 

: 

: 

In [ ]:
preprocess_to_npy(data, 'data_phoneme', vocab, max_len=512)

Tổng số mẫu: 2000
Đang cấp phát không gian cho data_phoneme_en.npy (0.00 GB)...


OSError: [Errno 22] Invalid argument: 'data_phoneme_en.npy'

: 

: 

: 

: 